In [1]:
def pq2tfmatrix(pq):
  import meshcat.transformations as tf
  import numpy as np
  """
  Convert a pq to a transformation matrix
  :param pq: [position, quaternion]
  :return: transformation matrix
  """
  p = pq[:3]
  q = pq[3:]
  q.insert(0, q.pop())
  return tf.quaternion_matrix(q) + np.array([[0, 0, 0, p[0]],
                                             [0, 0, 0, p[1]],
                                             [0, 0, 0, p[2]],
                                             [0, 0, 0, 0]])
    
def robotInit(numLinks, resource_path, displayInitJson, vis):
  import meshcat.geometry as g
  import numpy as np
  robot = vis['robot']
  partInitConfig = displayInitJson['part_init_config']
  for i in range(numLinks):
    robot[str(i)].set_transform(pq2tfmatrix(partInitConfig[i]))

  geometryPool = displayInitJson['geometry_pool']
  for i in range(len(geometryPool)):
    geometry = geometryPool[i]
    meshcatGeo = robot[str(geometry['part_id'])][str(geometry['geometry_id'])]
    meshcatGeo.set_transform(np.array(geometry['init_pm']).reshape(4, 4))
    if i % 2 == 1: 
      material = g.MeshPhongMaterial(color=0x0660FF)
    else:
      material = g.MeshPhongMaterial(color=0xD4D4D4)
    if i == 0:
      material = g.MeshPhongMaterial(color=0x755338)
    if(geometry['shape_type'] == 'box'):
      meshcatGeo.set_object(g.Box([geometry['length'], geometry['width'], geometry['height']]), material=material)
    elif(geometry['shape_type'] == 'capsule'):
      meshcatGeo.set_object(g.Cylinder(geometry['size'][0], geometry['size'][1]), material=material)
    elif(geometry['shape_type'] == 'sphere'):
      meshcatGeo.set_object(g.Sphere(geometry['radius']), material=material)
    elif(geometry['shape_type'] == 'mesh'):
      ext = geometry['resource_path'].split('.')[-1]
      if ext == 'stl':
        meshcatGeo.set_object(g.StlMeshGeometry.from_file(
          resource_path, geometry['resource_path']), material=material)
      elif ext == 'obj':
        meshcatGeo.set_object(g.ObjMeshGeometry.from_file(
          resource_path + geometry['resource_path']), material=material)
      else:
        print("Unknown mesh file type", ext)

def setRobotPq(numLinks, frame, pqs):
  robot = frame['robot']
  for i in range(numLinks):
    robot[str(i)].set_transform(pq2tfmatrix(pqs[i]))

def binarySearch(timeIndices, time):
  import math
  """
  Binary search to find the index of the closest time
  :param timeIndices: list of time indices
  :param time: target time
  :return: index of the closest time index
  """
  low = 0
  high = len(timeIndices) - 1
  while low <= high:
    mid = (low + high) // 2
    if math.isclose(timeIndices[mid], time):
      return mid
    if timeIndices[mid] < time:
      low = mid + 1
    else:
      high = mid - 1

  return min(int(low), len(timeIndices) - 1)

def animateRobotByRecords(numLinks, records, frameRate, vis):
  from meshcat.animation import Animation
  partpq = records['partPq']
  timeIndices = records['timeIndex']
  anim = Animation()
  anim.default_framerate = frameRate

  minTime = 0
  maxTime = timeIndices[-1]
  totalFrameNumber = int((maxTime - minTime) * anim.default_framerate)

  for i in range(totalFrameNumber):
    currentTime = minTime + i / anim.default_framerate
    currentIdx = binarySearch(timeIndices, currentTime)
    with anim.at_frame(vis, i) as frame:
      setRobotPq(numLinks, frame, partpq[currentIdx])

  vis.set_animation(anim)

In [2]:
import sys
sys.path.append("D:/code/sire/install/python/release")
import sire

In [3]:
from os.path import abspath
import os
cs = sire.ControlServer.instance()
print(abspath(os.getcwd()) + "/a1_modified.xml")
sire.fromXmlFile(cs, abspath(os.getcwd()) + "/a1_modified.xml")
cs.init()

d:\code\sire\demo\demo_python/a1_modified.xml


In [4]:
simulator = sire.simulator(cs)
model = cs.model()
target_q = [0, 0.9, -1.8, 0, 0.9, -1.8, 0, 0.9, -1.8, 0, 0.9, -1.8] # 这个才是对的角度，目前的角度都反了

while(not simulator.isTimeout() and not simulator.isEventListEmpty()):
  motionPool = model.motionPool()
  for i in range(12):
    motion = model.motionPool()[i]
    if isinstance(motion, sire.ActuatorSISO):
        motion.setDesiredValue(target_q[i])
  simulator.step(1, False)
  # print("Simulation time", simulator.simTime())

In [5]:
model = cs.model()
displayInitJson = model.displayInitJson()
result = simulator.recordsToJson()

In [6]:
import meshcat
displayInitJson
vis = meshcat.Visualizer()
resourcePath = "D:/code/sire/web_interface/public"
robotInit(model.numLinks(), resourcePath, displayInitJson, vis)
animateRobotByRecords(model.numLinks(), result, 1000, vis)
vis.jupyter_cell()

You can open the visualizer by visiting the following URL:
http://127.0.0.1:7001/static/


In [7]:
import matplotlib.pyplot as plt
timeIndices = result['timeIndex']
contactInfo = result["contactInfo"]
partpq = result["partPq"]
partvs = result["partVs"]
partas = result["partAs"]
dts = result["dts"]

lowerBound = binarySearch(timeIndices, 0)
upperBound = binarySearch(timeIndices, 4.4)

for i in range(lowerBound, upperBound + 1):
  print(i, timeIndices[i], dts[i], contactInfo[i])

0 0.0 0.0 None
1 0.001 0.001 None
2 0.002 0.001 None
3 0.003 0.001 None
4 0.004 0.001 None
5 0.005 0.001 None
6 0.006 0.001 None
7 0.007 0.001 None
8 0.008 0.001 None
9 0.009000000000000001 0.0010000000000000009 None
10 0.010000000000000002 0.0010000000000000009 None
11 0.011000000000000003 0.0010000000000000009 None
12 0.012000000000000004 0.0010000000000000009 None
13 0.013000000000000005 0.0010000000000000009 None
14 0.014000000000000005 0.0010000000000000009 None
15 0.015000000000000006 0.0010000000000000009 None
16 0.016000000000000007 0.0010000000000000009 None
17 0.017000000000000008 0.0010000000000000009 None
18 0.01800000000000001 0.0010000000000000009 None
19 0.01900000000000001 0.0010000000000000009 None
20 0.02000000000000001 0.0010000000000000009 None
21 0.02100000000000001 0.0010000000000000009 None
22 0.022000000000000013 0.0010000000000000009 None
23 0.023000000000000013 0.0010000000000000009 None
24 0.024000000000000014 0.0010000000000000009 None
25 0.02500000000000001

In [8]:
import json
with open("result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)